In [ ]:
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier

df = pd.read_csv("synthetic_dna_dataset.csv")
TARGET = "Class_Label"   # or "Disease_Risk"

y = LabelEncoder().fit_transform(df[TARGET])

kmer = CountVectorizer(analyzer="char", ngram_range=(3, 4), lowercase=False)
X_kmer = kmer.fit_transform(df["Sequence"])
X_kmer = X_kmer.multiply(1 / X_kmer.sum(axis=1)).tocsr()  # normalise by length

num_cols = ["GC_Content", "Sequence_Length", "kmer_3_freq", "Mutation_Flag"]
X = hstack([X_kmer, csr_matrix(df[num_cols].values)]).tocsr()

cv = StratifiedKFold(5, shuffle=True, random_state=42)
models = {
    "Baseline (majority)": DummyClassifier(strategy="most_frequent"),
    "LogReg": LogisticRegression(max_iter=3000),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1),
}
for name, m in models.items():
    print(f"{name:20s} {cross_val_score(m, X, y, cv=cv).mean():.3f}")